### Lab Assignment: Python Warmup and Logfile Analytics

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: January 23, 2026

---

This lab consists of two parts:

- Part 1 is the Python warmup
- Part 2 is the logfile analytics in PySpark

Answer the questions in this assignment, showing all code and solutions.

**Total points: 20**

---

### Part 1: Python Warmup

1) (1 PT) Rename this notebook to JupyterTutorial_[your_initials], where you will enter your initials in place of [your_initials].

2) (1 PT) In the cell below, enter a list of data science topics you find interesting.  Use the markdown style (you will need to change the style from the Code style).

insurance, risk modeling, fraud detection

3) (1 PT) In the cell below, enter the following Python list:  

some_vals = [1, 6, 10, 44]  

You will use the Code style, run the cell, and print the list.

In [1]:
some_vals = [1, 6, 10, 44]
print(some_vals)

[1, 6, 10, 44]


4) (1 PT) Use a list comprehension to return a filtered list containing only the values greater than 6.  
Call this list *some_vals_filtered* and print it.

In [2]:
some_vals_filtered = [x for x in some_vals if x > 6]
print(some_vals_filtered)

[10, 44]


Next, a small pandas dataframe is constructed.

In [4]:
import pandas as pd

df = pd.DataFrame({'first_name': ['Andy','Crystal'],
                   'domain_facebook' : [1,1],
                   'domain_foursquare' : [0,0],
                   'age' : [20, 32]})
df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,20
1,Crystal,1,0,32


5) (1 PT) In the cell below, write a list comprehension that returns the fields names in the dataframe `df` containing the string *domain*.  Run the cell to verify the correct result.

In [5]:
cols_domain = [x for x in df.columns if 'domain' in x]
print(cols_domain)

['domain_facebook', 'domain_foursquare']


6) (1 PT) Use the list comprehension from (5) to index into `df` and show the data for columns containing *domain*

In [6]:
df[cols_domain]

,domain_facebook,domain_foursquare
0,1,0
1,1,0


7) (1 PT) In the cell below, print the *domain_facebook* column

In [7]:
print(df["domain_facebook"])

0    1
1    1
Name: domain_facebook, dtype: int64


8) (1 PT) In the cell below, print the row with index 1.

In [8]:
print(df.loc[1])

first_name           Crystal
domain_facebook            1
domain_foursquare          0
age                       32
Name: 1, dtype: object


9) (1 PT) Next, you will cube the *age* column of `df` and assign the result to a new column called *agecube*.

Specifically, call the `apply` method with a `lambda function` inside to cube the *age* column.  
Print the dataframe.

In [9]:
df['agecube'] = df['age'].apply(lambda x: x**3)
print(df)

  first_name  domain_facebook  domain_foursquare  age  agecube
0       Andy                1                  0   20     8000
1    Crystal                1                  0   32    32768


10) (1 PT) Given the list of strings below, form one string, placing semicolons between each word.  It should look like this:  

`'the;quick;brown;fox'`

Print the resulting string.

In [11]:
some_list = ['the','quick','brown','fox']

In [12]:
print(";".join(some_list))

the;quick;brown;fox


---

### Part 2: Logfile Analytics

Import modules for Spark Session and regex 

Note: regexes can be used to search strings for patterns. Here is a [reference](https://realpython.com/regex-python/?utm_source=chatgpt.com).

In [18]:
from pyspark.sql import SparkSession
import re

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

11) (1 PT) Read in the logfile.txt data

In [19]:
data = sc.textFile("logfile.txt")

12) (1 PT) Count the number of rows of data

In [17]:
data.count()

100000

13) (2 PTS) Show the first five lines containing WARN.  
    Write code to count and print the total number of lines containing WARN.

In [21]:
line_count = data.filter(lambda line: 'WARN' in line)

line_count.take(5)

['2026-01-01T08:00:09.764000 [WARN] scheduler: heartbeat ip=92.174.240.101 latency_ms=475 trace=nw6i1m40mwvc seq=19',
 '2026-01-01T08:00:12.828000 [WARN] api: partition reassigned ip=84.161.222.95 latency_ms=4691 trace=yht26vhtm0eh seq=26',
 '2026-01-01T08:00:14.718000 [WARN] auth: job completed ip=230.79.10.241 latency_ms=3732 trace=tedo8mtq5qu9 seq=29',
 '2026-01-01T08:00:20.259000 [WARN] api: shuffle completed ip=212.54.74.42 latency_ms=4063 trace=ej2u544h7nwc seq=40',
 '2026-01-01T08:00:28.523000 [WARN] auth: cache miss ip=242.94.66.209 latency_ms=753 trace=feqvh0k4dwun seq=57']

In [22]:
line_count.count()

14823

14) (2 PTS) Write a word count program to count the number of each of these log levels:  

- WARN
- INFO
- DEBUG
- ERROR

In [24]:
word = ['WARN', 'INFO', 'DEBUG', 'ERROR']
for w in word: 
    print(w, data.filter(lambda x: w in x).count())

WARN 14823
INFO 70083
DEBUG 5143
ERROR 9951


15. (2 PTS) Return the three log lines with the highest latency. This is reported in the log as `latency_ms`.

Note: There may be more than three lines tied for highest latency, in which case, just show three records.

In [29]:
# create a function for latency 
def latency_filter(line): 
    return int(line.split('latency_ms=')[1].split()[0])

data_sorted_latency = data.sortBy(latency_filter, ascending = False)
data_sorted_latency.take(3)

['2026-01-01T08:30:03.360000 [INFO] api: heartbeat ip=154.9.232.228 latency_ms=5000 trace=e655i38fjysr seq=3785',
 '2026-01-01T08:37:27.714000 [INFO] metrics: request received ip=220.121.117.138 latency_ms=5000 trace=vtvn3oxagjp3 seq=4736',
 '2026-01-01T08:40:32.661000 [INFO] gateway: connection closed ip=85.231.101.84 latency_ms=5000 trace=3ml5tdora8on seq=5120']

16. (2 PTS) Compute the average latency for each service. Ignore log entries without a latency_ms.

In [32]:
def latency_ms(line): # check to ignore log entries without a latency_ms
    return 'latency_ms=' in line
    
def service_latency_filter(line): 
    service = line.split('] ')[1].split(':')[0]
    latency = int(line.split('latency_ms=')[1].split()[0])
    return (service, latency)
    
service_latency_pair = data.filter(latency_ms).map(service_latency_filter)

service_latency_groupedBy = service_latency_pair.groupByKey()

for service, latency in service_latency_groupedBy.collect(): 
    latency_list = list(latency)
    # avg
    print(service, sum(latency_list) / len(latency_list))



api 2503.894314115308
metrics 2485.1066211495713
auth 2506.9450742455692
storage 2505.294059730883
gateway 2492.284061901723
scheduler 2483.6426985888697
spark 2508.2136396381975
ingest 2501.4536561898653
